# Lab 5 — Part 4: Pipeline source and resource preview

ใช้ WSL/kernel `itcs355-lab3` เดิม แล้วรัน cell เดียว ไม่ต้องรัน Parts 1–3 ซ้ำ

พาร์ตนี้:
- ตรวจ estimate commit `83c400d` และ hash ที่บันทึกไว้
- ตรวจ data contract ของข้อมูล Lab 2 จริง และซ้อม deliberate contract failure
- ซ้อม gate ด้วยค่าทดสอบที่ระบุว่า synthetic; ยังไม่ใช่ผล training/cloud
- อ่าน datastores, input blob metadata, incumbent, CI identity และตรวจ image digest เดิมใน ACR
- สร้าง pipeline 6 nodes: ingest → validate → train → evaluate → register → deploy staging
- สร้าง variant ที่ฝึก DummyClassifier เพื่อทดสอบ gate fail ใน managed run ภายหลัง
- parse pipeline YAML ด้วย SDK ในเครื่อง และเก็บแผน identities/roles/resource ให้ตรวจ

**ไม่มี create/update/delete cloud resource, role grant, image build/push, training หรือ Git commit**

แผน compute คือ `lab5train` (DS2 v2) และ `lab5release` (DS1 v2) แยก identities
training identity ไม่อยู่บน release cluster และ CI identity ไม่อยู่บน training cluster
serving ใช้ identity ของตัวเอง ไม่ reuse training identity

Dedicated เป็น fallback ตาม quota; ข้อ discounted compute ยังไม่ผ่าน
งบ THB ทั้งเทอมยังต้อง reconcile — พาร์ตนี้ยังไม่เริ่มงานที่คิดเงิน

หากไม่มี progress ใหม่เกิน 5 นาที ให้ Interrupt และส่ง output ล่าสุด


In [2]:
from pathlib import Path
import platform
import subprocess
import sys

if platform.system() != "Linux" or "microsoft" not in platform.release().lower():
    raise RuntimeError("เลือก kernel itcs355-lab3 ฝั่ง WSL ก่อนครับ")
here = Path.cwd().resolve()
lab = next((base / "lab5" for base in (here, *here.parents)
            if (base / "lab5/scripts/prepare_pipeline.py").is_file()
            and (base / "lab4/README.md").is_file()), None)
if lab is None:
    raise RuntimeError("เปิด notebook จาก repository เดิมก่อนครับ")
print("Part 4: local rehearsal + read-only Azure checks + resource preview.", flush=True)
with subprocess.Popen(
    [sys.executable, "-u", "-m", "scripts.prepare_pipeline"], cwd=lab,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
) as task:
    try:
        for line in task.stdout:
            print(line, end="", flush=True)
        exit_code = task.wait()
    except KeyboardInterrupt:
        task.terminate()
        try:
            task.wait(timeout=10)
        except subprocess.TimeoutExpired:
            task.kill()
        raise
if exit_code:
    raise RuntimeError("Part 4 ยังไม่ผ่าน ส่ง error ด้านบนให้ผู้ช่วยตรวจ ไม่ต้องรันพาร์ตก่อนหน้าซ้ำครับ")
print("Part 4 เสร็จ: ส่ง SEND THIS SUMMARY แล้วหยุดรอตรวจ resource/role plan ครับ", flush=True)


Part 4: local rehearsal + read-only Azure checks + resource preview.


[13:51:25] 1/4 Verify committed estimate and previous observations
[13:51:26] 2/4 Rehearse the real data contract plus synthetic pass/fail gate probes
[13:51:26] 3/4 Read datastores, CI identity, ACR images and workspace tracking URI
[13:51:44] 4/4 Stage source allowlist, compile/parse six-step pipeline and write permission preview
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSche

## ผลและขั้นถัดไป

รายงาน: `lab5/reports/pipeline-preview.json`
มี source bundle hashes, DAG paths, gate policy และ role scopes จริง
local source bundles อยู่ใน `lab5/_bundles/` (ignored); สร้างใหม่ได้จาก source allowlist
ยังไม่ upload หรือ submit bundles เหล่านี้ และยังไม่ถือเป็น lineage ของ managed job ที่รันแล้ว

ส่ง summary กลับมา จากนั้นตรวจแผน source commit/สิทธิ์/งบ ก่อนเริ่ม managed execution
Gate margin 0.002 เป็น operational rule สำหรับสาธิต ไม่ใช่หลักฐานเชิงสถิติว่าผลดีขึ้นอย่างมั่นคง
